# GridSense-AZ — Graph WaveNet + Quantile Head Training

**Hero model** for the APS AI-for-Energy hackathon. Trains on Colab Pro (L4 GPU target) with a graceful local-CPU fallback for smoke tests.

Pipeline (single-file reproducible):
1. Install pinned deps
2. Imports + seed + config
3. Mount Drive (or local ckpt dir)
4. Clone repo into Colab runtime
5. Data loader — **dummy synthetic** mode + **real** mode stub
6. Topology loader (fixed adjacency) or random-sparse fallback
7. Instantiate `GraphWaveNetQuantile`
8. Train with PyTorch Lightning (Drive checkpoint, early stop, grad-clip)
9. Evaluate — MAE / RMSE / MAPE / pinball / coverage + plots
10. Save weights + write `reports/gwnet_v1.md` + (stubbed) HF upload
11. Captum Integrated-Gradients sanity check

> **Smoke mode** is on by default when `SMOKE=1` env var is set (or `smoke=True` below). It runs 2 epochs × tiny dims on CPU — the nbconvert job in CI uses this.

See `PLAN.md` §3.2 (model rationale), §3.3 (stack), §4 (datasets), §10 (DoD).


## Cell 1 — Install dependencies

Pinned to `PLAN.md` §3.3. Skipped when running locally against the uv venv.

In [1]:
import os, sys

IS_COLAB = "google.colab" in sys.modules
SKIP_PIP = os.environ.get("GRIDSENSE_SKIP_PIP", "0") == "1" or not IS_COLAB

if SKIP_PIP:
    print("[install] skipping pip install (not on Colab or GRIDSENSE_SKIP_PIP=1)")
else:
    # Torch 2.4+, PyG 2.6+, PL 2.4+ per PLAN §3.3. captum + hf_hub + gdown + wandb + duckdb.
    get_ipython().system(
        "pip install -q --upgrade "
        "'torch>=2.4,<2.6' "
        "'torch-geometric>=2.6,<2.7' "
        "'pytorch-lightning>=2.4,<2.6' "
        "'captum>=0.7' "
        "'huggingface_hub>=0.24' "
        "'gdown>=5.1' "
        "'wandb>=0.17' "
        "'duckdb>=1.0' "
        "'pandas>=2.2' "
        "'matplotlib>=3.8'"
    )
    print("[install] done")


[install] skipping pip install (not on Colab or GRIDSENSE_SKIP_PIP=1)


## Cell 2 — Imports, seed, config

In [2]:
from __future__ import annotations

import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Deterministic-ish.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

SMOKE = os.environ.get("GRIDSENSE_SMOKE", "0") == "1"

CFG = {
    "seed": SEED,
    "horizon": 24,
    "input_len": 168,
    "num_quantiles": 3,
    "quantile_levels": (0.1, 0.5, 0.9),
    "batch_size": 16,
    "epochs": 30,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "hidden_dim": 32,
    "num_blocks": 4,
    "kernel_size": 2,
    "dilation_growth": 2,
    "dropout": 0.3,
    "input_dim": 5,
    "num_nodes": 130,       # IEEE-123-ish; dummy mode override below
    "num_timesteps": 17520, # ~2 years hourly
    "mode": "dummy",
}

if SMOKE:
    CFG.update({
        "epochs": 2,
        "batch_size": 4,
        "num_nodes": 20,
        "num_timesteps": 2000,
        "hidden_dim": 16,
        "num_blocks": 3,
    })

print(f"[config] SMOKE={SMOKE}")
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))


[config] SMOKE=True
{
  "seed": "42",
  "horizon": "24",
  "input_len": "168",
  "num_quantiles": "3",
  "quantile_levels": "(0.1, 0.5, 0.9)",
  "batch_size": "4",
  "epochs": "2",
  "lr": "0.001",
  "weight_decay": "0.0001",
  "hidden_dim": "16",
  "num_blocks": "3",
  "kernel_size": "2",
  "dilation_growth": "2",
  "dropout": "0.3",
  "input_dim": "5",
  "num_nodes": "20",
  "num_timesteps": "2000",
  "mode": "dummy"
}


## Cell 3 — Mount Drive (or local ckpt fallback)

Colab sessions can die; checkpointing to Drive is non-negotiable. Locally, we fall back to `./ckpt_local`.

In [3]:
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    CKPT_DIR = Path("/content/drive/MyDrive/jarvis/gridsense-az/ckpt")
    print("[drive] mounted")
except Exception as e:
    # Local fallback: write into the current directory's ckpt_local/.
    CKPT_DIR = Path.cwd() / "ckpt_local"
    print(f"[drive] fallback to local: {CKPT_DIR} (reason: {type(e).__name__})")

CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("CKPT_DIR =", CKPT_DIR)


[drive] fallback to local: /home/divyansh/Downloads/hackathon/energy/notebooks/ckpt_local (reason: ModuleNotFoundError)
CKPT_DIR = /home/divyansh/Downloads/hackathon/energy/notebooks/ckpt_local


## Cell 4 — Clone the repo into the Colab runtime

Locally we just inject `../src` onto `sys.path`.

In [4]:
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    repo_path = Path("/content/repo")
    if not repo_path.exists():
        get_ipython().system(
            "git clone --depth 1 https://github.com/dc-ai-labs/gridsense-az.git /content/repo"
        )
    sys.path.insert(0, "/content/repo/src")
    REPO_ROOT = repo_path
else:
    # notebooks/ sits next to src/; resolve repo root relative to this file.
    try:
        here = Path(__file__).resolve().parent  # type: ignore[name-defined]
    except NameError:
        here = Path.cwd()
    # If launched via nbconvert from repo root, cwd is the repo root; if launched
    # from notebooks/, go up one level.
    REPO_ROOT = here if (here / "src" / "gridsense").exists() else here.parent
    src_path = str(REPO_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print("REPO_ROOT =", REPO_ROOT)
from gridsense.models.gwnet import GraphWaveNetQuantile, pinball_loss  # noqa: E402
print("[repo] gridsense.models.gwnet imported OK")


REPO_ROOT = /home/divyansh/Downloads/hackathon/energy
[repo] gridsense.models.gwnet imported OK


## Cell 5 — Data loader (dummy + real stub)

**Dummy mode** generates a plausible hourly load time series with:
- hourly + weekly + seasonal cycles (sin/cos encoded),
- a standardized temperature feature peaking at hour-18 each day,
- per-node load shaped by a fixed but random mixing of all 5 features (nonlinear + noise).

**Real mode** is a stub — it will load parquet from `data/features/` once the data pipeline lands.

Produces a chronological 70/15/15 train/val/test split (no leakage).

In [5]:
def _synthetic_features(num_timesteps: int, num_nodes: int, input_dim: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    t = np.arange(num_timesteps)
    hour = t % 24
    dow = (t // 24) % 7
    doy = (t // 24) % 365

    # Per-timestep global features (broadcast across nodes below).
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    dow_sin = np.sin(2 * np.pi * dow / 7)
    dow_cos = np.cos(2 * np.pi * dow / 7)

    # Temperature: seasonal baseline + daily cycle peaking at hour 18 + weekly wobble + noise.
    seasonal = 10 * np.sin(2 * np.pi * doy / 365 - np.pi / 2)   # [-10, +10]
    daily = 8 * np.sin(2 * np.pi * (hour - 18) / 24 + np.pi / 2)  # peak at 18:00
    weekly = 2 * np.sin(2 * np.pi * dow / 7)
    temp_c = 25.0 + seasonal + daily + weekly + rng.normal(0, 1.0, size=num_timesteps)
    temp_scaled = (temp_c - temp_c.mean()) / (temp_c.std() + 1e-6)

    # Stack global features to [T, F_global]. F_global = 5 matches CFG["input_dim"].
    F_global = np.stack([hour_sin, hour_cos, dow_sin, dow_cos, temp_scaled], axis=1).astype(np.float32)
    assert F_global.shape[1] == input_dim, "adjust input_dim or feature list"

    # Broadcast to per-node features [T, N, F]. In the real pipeline these become
    # per-node weather + calendar joined to the topology.
    feats = np.broadcast_to(F_global[:, None, :], (num_timesteps, num_nodes, input_dim)).copy()

    # Targets: each node has a fixed random mixing vector over features, plus a
    # nonlinear kink on temperature (AC load rises above 30 C), plus noise.
    mix = rng.normal(0, 1.0, size=(num_nodes, input_dim)).astype(np.float32)
    # One scalar target per (t, n): dot per-node feature vector with per-node mix.
    targets = np.einsum("tnf,nf->tn", feats, mix).astype(np.float32)

    # Nonlinear AC bump: ReLU(temp - tau) scaled by a per-node coefficient.
    ac_coef = rng.uniform(0.5, 2.0, size=num_nodes).astype(np.float32)
    ac_bump = np.maximum(temp_c[:, None] - 30.0, 0.0) * ac_coef[None, :]
    targets = targets + ac_bump.astype(np.float32)
    targets = targets + rng.normal(0, 0.3, size=targets.shape).astype(np.float32)
    return feats.astype(np.float32), targets.astype(np.float32)


def load_features(data_root: str | Path, mode: str = "real"):
    """Return (features [T,N,F], targets [T,N]).

    Parameters
    ----------
    data_root : path to the repo's data/ directory (used only in real mode).
    mode : "real" -> load parquet from `data/features/`. "dummy" -> synthesize.
    """
    if mode == "dummy":
        return _synthetic_features(
            num_timesteps=CFG["num_timesteps"],
            num_nodes=CFG["num_nodes"],
            input_dim=CFG["input_dim"],
            seed=CFG["seed"],
        )
    if mode == "real":
        data_root = Path(data_root)
        feats_pq = data_root / "features" / "feeder_load_timeseries.parquet"
        raise NotImplementedError(
            f"real mode: implement parquet load from {feats_pq}. "
            "Blocked on data pipeline (PLAN §4)."
        )
    raise ValueError(f"unknown mode {mode!r}")


class WindowedDataset(Dataset):
    """Sliding-window (input_len -> horizon) pairs, chronologically ordered."""

    def __init__(self, feats: np.ndarray, targets: np.ndarray, input_len: int, horizon: int):
        assert feats.shape[0] == targets.shape[0]
        self.feats = feats
        self.targets = targets
        self.input_len = input_len
        self.horizon = horizon
        self.n = feats.shape[0] - input_len - horizon + 1
        if self.n <= 0:
            raise ValueError(
                f"not enough timesteps ({feats.shape[0]}) for input_len={input_len} + horizon={horizon}"
            )

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, i: int):
        x = self.feats[i : i + self.input_len]                # [T_in, N, F]
        y = self.targets[i + self.input_len : i + self.input_len + self.horizon]  # [H, N]
        return torch.from_numpy(x), torch.from_numpy(y)


def chronological_split(feats: np.ndarray, targets: np.ndarray, ratios=(0.70, 0.15, 0.15)):
    assert abs(sum(ratios) - 1.0) < 1e-6
    t = feats.shape[0]
    a = int(t * ratios[0])
    b = a + int(t * ratios[1])
    return (feats[:a], targets[:a]), (feats[a:b], targets[a:b]), (feats[b:], targets[b:])


# Build datasets in the configured mode.
feats, targets = load_features(REPO_ROOT / "data", mode=CFG["mode"])
print("[data] features:", feats.shape, "targets:", targets.shape)

# Standardize targets for stable training (we track the scale for inverse).
target_mean = float(targets.mean())
target_std = float(targets.std() + 1e-6)
targets_std = (targets - target_mean) / target_std
print(f"[data] target_mean={target_mean:.3f}  target_std={target_std:.3f}")

(tr_f, tr_y), (va_f, va_y), (te_f, te_y) = chronological_split(feats, targets_std)
ds_tr = WindowedDataset(tr_f, tr_y, CFG["input_len"], CFG["horizon"])
ds_va = WindowedDataset(va_f, va_y, CFG["input_len"], CFG["horizon"])
ds_te = WindowedDataset(te_f, te_y, CFG["input_len"], CFG["horizon"])
print(f"[data] train={len(ds_tr)} val={len(ds_va)} test={len(ds_te)}")

num_workers = 0  # keep 0 for reproducibility + CPU-only smoke runs
dl_tr = DataLoader(ds_tr, batch_size=CFG["batch_size"], shuffle=True, num_workers=num_workers)
dl_va = DataLoader(ds_va, batch_size=CFG["batch_size"], shuffle=False, num_workers=num_workers)
dl_te = DataLoader(ds_te, batch_size=CFG["batch_size"], shuffle=False, num_workers=num_workers)

# Smoke-mode also caps batches per epoch for speed.
LIMIT_TRAIN_BATCHES = 20 if SMOKE else 1.0
LIMIT_VAL_BATCHES = 10 if SMOKE else 1.0


[data] features: (2000, 20, 5) targets: (2000, 20)
[data] target_mean=0.029  target_std=1.622
[data] train=1209 val=109 test=109


## Cell 6 — Topology loader (IEEE-123) with random fallback

Tries `gridsense.topology.load_ieee123(...)` first. If the `.dss` files aren't on disk or the helper isn't implemented yet, we build a random sparse binary adjacency so the model still has a fixed graph to mix with the learned adaptive one.

In [6]:
feeder_dir = REPO_ROOT / "data" / "raw" / "ieee123"

adj = None
try:
    from gridsense.topology import load_ieee123  # type: ignore
    adj = load_ieee123(str(feeder_dir))
    print("[topology] loaded IEEE-123 adjacency:", tuple(adj.shape))
except Exception as e:
    print(f"[topology] fallback to random sparse adjacency (reason: {type(e).__name__}: {e})")
    rng = np.random.default_rng(CFG["seed"])
    n = CFG["num_nodes"]
    # Random ~k-regular-ish sparse graph (k=3 neighbors on average per node).
    k = max(2, int(round(np.log2(n) + 1)))
    idx = rng.integers(0, n, size=(n, k))
    adj_np = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in idx[i]:
            if i != j:
                adj_np[i, j] = 1.0
                adj_np[j, i] = 1.0  # symmetrize
    adj = torch.from_numpy(adj_np)

assert adj.shape == (CFG["num_nodes"], CFG["num_nodes"]), (
    f"adj shape {tuple(adj.shape)} != num_nodes={CFG['num_nodes']} — check CFG"
)
print("[topology] adj nnz =", int((adj > 0).sum().item()))


[topology] fallback to random sparse adjacency (reason: ImportError: cannot import name 'load_ieee123' from 'gridsense.topology' (/home/divyansh/Downloads/hackathon/energy/src/gridsense/topology.py))
[topology] adj nnz = 158


## Cell 7 — Instantiate `GraphWaveNetQuantile`

In [7]:
model = GraphWaveNetQuantile(
    num_nodes=CFG["num_nodes"],
    adj_init=adj,
    input_dim=CFG["input_dim"],
    hidden_dim=CFG["hidden_dim"],
    num_blocks=CFG["num_blocks"],
    kernel_size=CFG["kernel_size"],
    dilation_growth=CFG["dilation_growth"],
    horizon=CFG["horizon"],
    quantile_levels=CFG["quantile_levels"],
    dropout=CFG["dropout"],
)

n_params = sum(p.numel() for p in model.parameters())
print(f"[model] params = {n_params:,}")
print(f"[model] receptive_field = {model.receptive_field} (input_len = {CFG['input_len']})")


[model] params = 29,336
[model] receptive_field = 8 (input_len = 168)


## Cell 8 — PyTorch Lightning training loop

- Adam, LR 1e-3, weight decay 1e-4
- Pinball (quantile) loss
- `ReduceLROnPlateau` on val loss
- `ModelCheckpoint` → top-3 saved to Drive each epoch
- `EarlyStopping` patience=5
- `gradient_clip_val=5.0`

`SMOKE` flag caps epochs + batches so the whole notebook finishes in ~a couple minutes on CPU.

In [8]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


class GWNetLightning(pl.LightningModule):
    def __init__(self, model: nn.Module, cfg: dict):
        super().__init__()
        self.model = model
        self.cfg = cfg
        self.quantiles = tuple(cfg["quantile_levels"])

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage: str):
        x, y = batch
        pred = self(x)  # [B, H, N, Q]
        loss = pinball_loss(pred, y, self.quantiles)
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, _):
        return self._step(batch, "train")

    def validation_step(self, batch, _):
        return self._step(batch, "val")

    def configure_optimizers(self):
        opt = torch.optim.Adam(
            self.parameters(), lr=self.cfg["lr"], weight_decay=self.cfg["weight_decay"]
        )
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "monitor": "val_loss", "interval": "epoch"},
        }


lit = GWNetLightning(model, CFG)

ckpt_cb = ModelCheckpoint(
    dirpath=str(CKPT_DIR),
    filename="gwnet-epoch{epoch:02d}-val{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=3,
    every_n_epochs=1,
    save_last=True,
)
early_cb = EarlyStopping(monitor="val_loss", mode="min", patience=5)

accel = "gpu" if torch.cuda.is_available() else "cpu"
print(f"[train] accelerator = {accel}")

trainer = pl.Trainer(
    max_epochs=CFG["epochs"],
    accelerator=accel,
    devices=1,
    gradient_clip_val=5.0,
    callbacks=[ckpt_cb, early_cb],
    limit_train_batches=LIMIT_TRAIN_BATCHES,
    limit_val_batches=LIMIT_VAL_BATCHES,
    log_every_n_steps=1,
    enable_progress_bar=True,
    default_root_dir=str(CKPT_DIR.parent / "pl_logs"),
    enable_model_summary=False,
)

t0 = time.time()
trainer.fit(lit, dl_tr, dl_va)
train_time = time.time() - t0
print(f"[train] done in {train_time:.1f}s  best ckpt = {ckpt_cb.best_model_path}")


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


/home/divyansh/Downloads/hackathon/energy/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

/home/divyansh/Downloads/hackathon/energy/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py
:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/divyansh/Downloads/hackathon/energy/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/d
ata_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

[train] accelerator = cpu


/home/divyansh/Downloads/hackathon/energy/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/d
ata_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=2` reached.


[train] done in 14.9s  best ckpt = /home/divyansh/Downloads/hackathon/energy/notebooks/ckpt_local/gwnet-epochepoch=01-valval_loss=0.2489.ckpt


## Cell 9 — Evaluation on the held-out test split

- MAE, RMSE, MAPE on the p50 forecast (on original unstandardized scale)
- Pinball loss — overall and per quantile
- Coverage: share of test points with `p10 <= y <= p90` (target ≥0.8)
- Plots: reliability diagram + sample 24 h ribbon


In [9]:
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt

lit.eval()
device = next(lit.parameters()).device
preds_all, tgt_all = [], []
with torch.no_grad():
    for xb, yb in dl_te:
        xb = xb.to(device)
        p = lit(xb).cpu().numpy()
        preds_all.append(p)
        tgt_all.append(yb.numpy())
preds = np.concatenate(preds_all, axis=0)   # [N_test, H, N_nodes, Q]
tgts = np.concatenate(tgt_all, axis=0)      # [N_test, H, N_nodes]
print("[eval] preds:", preds.shape, "targets:", tgts.shape)

# Inverse-standardize back to original units.
preds_orig = preds * target_std + target_mean
tgts_orig = tgts * target_std + target_mean

p10 = preds_orig[..., 0]
p50 = preds_orig[..., 1]
p90 = preds_orig[..., 2]

mae = float(np.mean(np.abs(p50 - tgts_orig)))
rmse = float(np.sqrt(np.mean((p50 - tgts_orig) ** 2)))
# MAPE — guard against tiny denominators.
denom = np.where(np.abs(tgts_orig) < 1e-3, 1e-3, tgts_orig)
mape = float(np.mean(np.abs((p50 - tgts_orig) / denom)))

# Pinball — overall + per quantile. Use standardized-scale preds for loss consistency.
pin_overall = float(pinball_loss(
    torch.from_numpy(preds), torch.from_numpy(tgts), CFG["quantile_levels"]
).item())
pin_per_q: list[float] = []
for qi, q in enumerate(CFG["quantile_levels"]):
    pin_per_q.append(float(pinball_loss(
        torch.from_numpy(preds[..., qi:qi + 1]),
        torch.from_numpy(tgts),
        (q,),
    ).item()))

# Coverage — target in [p10, p90].
cov_80 = float(np.mean((tgts_orig >= p10) & (tgts_orig <= p90)))

print(f"[eval] MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.4f}")
print(f"[eval] pinball overall={pin_overall:.4f}  per-q={[round(x,4) for x in pin_per_q]}")
print(f"[eval] 80%-interval coverage={cov_80:.3f}  (target >= 0.80)")

# Plots dir.
PLOTS_DIR = REPO_ROOT / "reports" / "figures"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Reliability diagram -----------------------------------------------------
# Predict quantiles across a sweep; check empirical vs nominal.
nominal = np.linspace(0.05, 0.95, 10)
# Approximate calibration: interpolate predicted CDF using the 3 pinned quantiles.
# Simple way: empirical coverage at p10/p50/p90 levels (one-sided).
nom_onesided = np.array([0.10, 0.50, 0.90])
emp = []
for qi, q in enumerate(CFG["quantile_levels"]):
    emp.append(float(np.mean(tgts_orig <= preds_orig[..., qi])))
emp = np.array(emp)

fig, ax = plt.subplots(figsize=(4, 4))
ax.plot([0, 1], [0, 1], "--", color="gray", label="ideal")
ax.scatter(nom_onesided, emp, s=80, color="C0", zorder=3, label="observed")
for q, e in zip(nom_onesided, emp):
    ax.annotate(f"  ({q:.2f}, {e:.2f})", (q, e))
ax.set_xlabel("nominal quantile")
ax.set_ylabel("empirical share below predicted")
ax.set_title("Reliability — quantile calibration")
ax.legend()
fig.tight_layout()
reliability_png = PLOTS_DIR / "gwnet_v1_reliability.png"
fig.savefig(reliability_png, dpi=120)
plt.close(fig)

# --- Sample 24 h forecast ribbon --------------------------------------------
# Pick a middle test window and a node with healthy variance.
sample_idx = min(len(ds_te) // 2, preds_orig.shape[0] - 1)
node_idx = int(np.argmax(tgts_orig[sample_idx].std(axis=0)))
hours = np.arange(CFG["horizon"])
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.fill_between(
    hours,
    p10[sample_idx, :, node_idx],
    p90[sample_idx, :, node_idx],
    alpha=0.25, color="C0", label="p10-p90",
)
ax.plot(hours, p50[sample_idx, :, node_idx], color="C0", label="p50")
ax.plot(hours, tgts_orig[sample_idx, :, node_idx], color="black", linestyle="--", label="actual")
ax.set_xlabel("hour ahead")
ax.set_ylabel("load (dummy scale)")
ax.set_title(f"24h forecast ribbon — node {node_idx}")
ax.legend()
fig.tight_layout()
ribbon_png = PLOTS_DIR / "gwnet_v1_ribbon.png"
fig.savefig(ribbon_png, dpi=120)
plt.close(fig)

# --- Loss curve --------------------------------------------------------------
# Lightning writes metrics into pl_logs/version_x/metrics.csv once per epoch.
loss_png = PLOTS_DIR / "gwnet_v1_losscurve.png"
try:
    import pandas as pd

    pl_logs_dir = Path(trainer.logger.log_dir) if trainer.logger else None
    metrics_csv = pl_logs_dir / "metrics.csv" if pl_logs_dir else None
    if metrics_csv and metrics_csv.exists():
        df = pd.read_csv(metrics_csv)
        fig, ax = plt.subplots(figsize=(5, 3))
        if "train_loss" in df:
            tr = df.dropna(subset=["train_loss"])
            ax.plot(tr["epoch"], tr["train_loss"], label="train")
        if "val_loss" in df:
            va = df.dropna(subset=["val_loss"])
            ax.plot(va["epoch"], va["val_loss"], label="val")
        ax.set_xlabel("epoch")
        ax.set_ylabel("pinball loss")
        ax.set_title("Loss curve")
        ax.legend()
        fig.tight_layout()
        fig.savefig(loss_png, dpi=120)
        plt.close(fig)
    else:
        # Fallback placeholder.
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.text(0.5, 0.5, "no metrics.csv found", ha="center", va="center")
        ax.set_axis_off()
        fig.tight_layout()
        fig.savefig(loss_png, dpi=120)
        plt.close(fig)
except Exception as e:
    print("[plot] loss curve skipped:", e)

print(f"[plot] saved to {PLOTS_DIR}")


[W418 17:16:23.460145679 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.


[eval] preds: (109, 24, 20, 3) targets: (109, 24, 20)
[eval] MAE=1.3836  RMSE=1.8749  MAPE=1.2653
[eval] pinball overall=0.2814  per-q=[0.19, 0.4265, 0.2278]
[eval] 80%-interval coverage=0.786  (target >= 0.80)


[plot] saved to /home/divyansh/Downloads/hackathon/energy/reports/figures


## Cell 10 — Save weights + report + (stubbed) HF upload

Writes:
- `ckpt/gwnet_v1.pt` on Drive (or local fallback)
- `models/gwnet_v1.pt` inside the repo
- `reports/gwnet_v1_metrics.json`
- `reports/gwnet_v1.md` with embedded figures
- HF upload call — **commented / stubbed** until the real training pass.

In [10]:
MODEL_VERSION = "v1"

weights_drive = CKPT_DIR / f"gwnet_{MODEL_VERSION}.pt"
torch.save(model.state_dict(), weights_drive)
print(f"[save] drive weights -> {weights_drive}")

weights_local = REPO_ROOT / "models" / f"gwnet_{MODEL_VERSION}.pt"
weights_local.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), weights_local)
print(f"[save] repo weights  -> {weights_local}")

# Metrics JSON.
metrics = {
    "version": MODEL_VERSION,
    "mode": CFG["mode"],
    "smoke": SMOKE,
    "accelerator": accel,
    "train_time_s": round(train_time, 2),
    "epochs_ran": int(trainer.current_epoch) + 1,
    "best_ckpt": str(ckpt_cb.best_model_path),
    "mae_p50": mae,
    "rmse_p50": rmse,
    "mape_p50": mape,
    "pinball_overall": pin_overall,
    "pinball_per_quantile": dict(zip([str(q) for q in CFG["quantile_levels"]], pin_per_q)),
    "coverage_80": cov_80,
    "num_params": n_params,
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
}
metrics_json = REPO_ROOT / "reports" / f"gwnet_{MODEL_VERSION}_metrics.json"
metrics_json.parent.mkdir(parents=True, exist_ok=True)
metrics_json.write_text(json.dumps(metrics, indent=2))
print(f"[save] metrics       -> {metrics_json}")

# Markdown report.
report = f"""# Graph WaveNet v1 — training report ({'SMOKE' if SMOKE else 'FULL'})

- **mode:** `{CFG["mode"]}`
- **accelerator:** `{accel}`
- **train time:** {train_time:.1f}s (epochs ran: {int(trainer.current_epoch)+1})
- **params:** {n_params:,}
- **best ckpt:** `{ckpt_cb.best_model_path}`

## Metrics (p50 on original scale)

| metric | value |
|---|---|
| MAE  | {mae:.4f} |
| RMSE | {rmse:.4f} |
| MAPE | {mape:.4f} |
| pinball (overall) | {pin_overall:.4f} |
| 80%-interval coverage | {cov_80:.3f} |

Per-quantile pinball: {dict(zip([str(q) for q in CFG['quantile_levels']], [round(x,4) for x in pin_per_q]))}

## Plots

![loss curve](figures/gwnet_v1_losscurve.png)

![reliability](figures/gwnet_v1_reliability.png)

![sample ribbon](figures/gwnet_v1_ribbon.png)

## Notes

Run config:

```json
{json.dumps(metrics['config'], indent=2)}
```
"""
report_md = REPO_ROOT / "reports" / f"gwnet_{MODEL_VERSION}.md"
report_md.write_text(report)
print(f"[save] report        -> {report_md}")

# HF upload — STUBBED. Uncomment + set HF_TOKEN for the real training pass.
HF_REPO = "dchanda/gridsense-az"
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi, create_repo
        # create_repo(HF_REPO, private=False, exist_ok=True, token=HF_TOKEN)
        # HfApi().upload_file(
        #     path_or_fileobj=str(weights_local),
        #     path_in_repo=f"gwnet_{MODEL_VERSION}.pt",
        #     repo_id=HF_REPO,
        #     token=HF_TOKEN,
        # )
        print(f"[hf] (stubbed) would upload {weights_local.name} -> {HF_REPO}")
    except Exception as e:
        print(f"[hf] upload skipped: {e}")
else:
    print("[hf] HF_TOKEN not set — skipping upload stub")


[save] drive weights -> /home/divyansh/Downloads/hackathon/energy/notebooks/ckpt_local/gwnet_v1.pt
[save] repo weights  -> /home/divyansh/Downloads/hackathon/energy/models/gwnet_v1.pt
[save] metrics       -> /home/divyansh/Downloads/hackathon/energy/reports/gwnet_v1_metrics.json
[save] report        -> /home/divyansh/Downloads/hackathon/energy/reports/gwnet_v1.md
[hf] HF_TOKEN not set — skipping upload stub


## Cell 11 — Integrated Gradients explainability sanity check

For 3 random (bus, timestep) points in the test split, attribute the predicted p50 back to the input features via Captum IG. We save the top-5 features by absolute attribution per point as a bar chart.

This is a **sanity check**, not the final driver panel — the dashboard will run IG on the full test set.

In [11]:
try:
    from captum.attr import IntegratedGradients
    HAS_CAPTUM = True
except Exception as e:
    print("[captum] import failed — skipping IG:", e)
    HAS_CAPTUM = False

if HAS_CAPTUM:
    FEATURE_NAMES = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "temp_scaled"]

    # Wrapper that maps [B, T_in, N, F] -> scalar for a chosen (horizon_step, node).
    class Scalarize(nn.Module):
        def __init__(self, model, h: int, n: int, q_idx: int = 1):
            super().__init__()
            self.model = model
            self.h = h
            self.n = n
            self.q_idx = q_idx  # 0->p10, 1->p50, 2->p90

        def forward(self, x):
            y = self.model(x)  # [B, H, N, Q]
            return y[:, self.h, self.n, self.q_idx]  # [B]

    lit.eval()
    ig_samples = []
    rng = np.random.default_rng(CFG["seed"])
    for k in range(3):
        idx = int(rng.integers(0, len(ds_te)))
        node = int(rng.integers(0, CFG["num_nodes"]))
        hstep = int(rng.integers(0, CFG["horizon"]))
        x, _ = ds_te[idx]
        x = x.unsqueeze(0).to(next(lit.parameters()).device).requires_grad_(True)
        wrap = Scalarize(lit.model, h=hstep, n=node).to(x.device)
        ig = IntegratedGradients(wrap)
        attributions = ig.attribute(x, baselines=torch.zeros_like(x), n_steps=16)
        # Reduce over (T_in, N) -> per-feature importance for this (idx, node, hstep).
        per_feat = attributions[0].detach().abs().mean(dim=(0, 1)).cpu().numpy()  # [F]
        order = np.argsort(per_feat)[::-1][:5]
        ig_samples.append({
            "idx": idx, "node": node, "hstep": hstep,
            "top": [(FEATURE_NAMES[i], float(per_feat[i])) for i in order],
        })

    # Plot the top-5 feature importances per sample.
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, s in zip(axes, ig_samples):
        names = [t[0] for t in s["top"]]
        vals = [t[1] for t in s["top"]]
        ax.barh(names[::-1], vals[::-1])
        ax.set_title(f"idx={s['idx']} node={s['node']} h={s['hstep']}")
        ax.set_xlabel("|IG|")
    fig.suptitle("Integrated Gradients — sanity check (3 random points)")
    fig.tight_layout()
    ig_png = PLOTS_DIR / "gwnet_v1_ig_sanity.png"
    fig.savefig(ig_png, dpi=120)
    plt.close(fig)

    # Append IG section to the report.
    with open(report_md, "a") as f:
        f.write("\n## Integrated Gradients (3 random points)\n\n")
        f.write(f"![ig sanity](figures/{ig_png.name})\n\n")
        for s in ig_samples:
            f.write(f"- idx={s['idx']} node={s['node']} hstep={s['hstep']}: top={s['top']}\n")

    print(f"[ig] saved -> {ig_png}")
    print(json.dumps(ig_samples, indent=2))
else:
    print("[ig] skipped")

print("\n[done] notebook finished.")


[ig] saved -> /home/divyansh/Downloads/hackathon/energy/reports/figures/gwnet_v1_ig_sanity.png
[
  {
    "idx": 9,
    "node": 15,
    "hstep": 15,
    "top": [
      [
        "dow_sin",
        2.038746424659621e-05
      ],
      [
        "hour_sin",
        1.9336099285283126e-05
      ],
      [
        "temp_scaled",
        1.3087155821267515e-05
      ],
      [
        "hour_cos",
        1.1410761544539127e-05
      ],
      [
        "dow_cos",
        3.0811565920885187e-06
      ]
    ]
  },
  {
    "idx": 47,
    "node": 8,
    "hstep": 20,
    "top": [
      [
        "temp_scaled",
        3.504312189761549e-05
      ],
      [
        "dow_sin",
        1.2516071365098469e-05
      ],
      [
        "hour_sin",
        1.043004340317566e-05
      ],
      [
        "dow_cos",
        3.51568291989679e-06
      ],
      [
        "hour_cos",
        3.3499870824016398e-06
      ]
    ]
  },
  {
    "idx": 9,
    "node": 13,
    "hstep": 4,
    "top": [
      [
       